<a href="https://colab.research.google.com/github/parth-electron/All-Projects/blob/main/Cifar-10%20Image%20Classification%20using%20CNN/Assignmentx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:


import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

# --------------------------------------------------------------------------
# Task 1: Data Understanding
# --------------------------------------------------------------------------

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Training data shape :", X_train.shape)  # (50000, 32, 32, 3)
print("Training labels shape:", y_train.shape)  # (50000, 1)
print("Test data shape     :", X_test.shape)    # (10000, 32, 32, 3)
print("Test labels shape   :", y_test.shape)     # (10000, 1)
print("Number of classes   :", len(CLASS_NAMES))
print("Class names         :", CLASS_NAMES)

# Show the first 5 training images with their labels
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i])
    ax.set_title(CLASS_NAMES[int(y_train[i])])
    ax.axis("off")
plt.tight_layout()
plt.savefig("sample_images.png", dpi=150)
print("\nSaved first-five sample images to sample_images.png")

# --------------------------------------------------------------------------
# Task 2: Data Preprocessing
# --------------------------------------------------------------------------

# Check for missing/corrupt values (CIFAR-10 is a clean, complete dataset,
# but we verify anyway as good practice)
print("\nAny NaNs in training images?", np.isnan(X_train).any())
print("Any NaNs in test images?    ", np.isnan(X_test).any())

# Normalize pixel values from [0, 255] to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# One-hot encode the labels (10 classes)
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

# Keras already provides a fixed 50,000 / 10,000 (train/test) split for
# CIFAR-10, which serves the 80/20-style train/test purpose of this task.
print(f"\nTraining set size: {X_train.shape[0]} images")
print(f"Testing set size : {X_test.shape[0]} images")

# --------------------------------------------------------------------------
# Task 3: Model Development (CNN)
# --------------------------------------------------------------------------

model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

EPOCHS = 20
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train_cat,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)

# --------------------------------------------------------------------------
# Task 4: Model Evaluation
# --------------------------------------------------------------------------

test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nTest Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

print("\n=== Classification report ===")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Training curves: accuracy and loss vs. epoch
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(history.history["accuracy"], label="Train Accuracy")
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Validation Loss")
axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
print("\nSaved training curves to training_curves.png")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(9, 8))
disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title("Confusion Matrix -- CIFAR-10 CNN")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
print("Saved confusion matrix to confusion_matrix.png")

with open("metrics.txt", "w") as f:
    f.write(f"test_loss,{test_loss:.4f}\n")
    f.write(f"test_accuracy,{test_accuracy:.4f}\n")

print("\nDone.")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3831s 22us/step
Training data shape : (50000, 32, 32, 3)
Training labels shape: (50000, 1)
Test data shape     : (10000, 32, 32, 3)
Test labels shape   : (10000, 1)
Number of classes   : 10
Class names         : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


/tmp/ipykernel_671/3435603933.py:53: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ax.set_title(CLASS_NAMES[int(y_train[i])])



Saved first-five sample images to sample_images.png

Any NaNs in training images? False
Any NaNs in test images?     False

Training set size: 50000 images
Testing set size : 10000 images


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 667,434 (2.55 MB)

 Trainable params: 666,986 (2.54 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/20
704/704 - 272s - 387ms/step - accuracy: 0.3502 - loss: 1.7868 - val_accuracy: 0.5150 - val_loss: 1.3389
Epoch 2/20
704/704 - 322s - 458ms/step - accuracy: 0.5118 - loss: 1.3577 - val_accuracy: 0.5840 - val_loss: 1.2073
Epoch 3/20
704/704 - 276s - 393ms/step - accuracy: 0.5900 - loss: 1.1539 - val_accuracy: 0.5908 - val_loss: 1.2858
Epoch 4/20
704/704 - 276s - 392ms/step - accuracy: 0.6402 - loss: 1.0287 - val_accuracy: 0.6762 - val_loss: 0.9348
Epoch 5/20
704/704 - 323s - 458ms/step - accuracy: 0.6720 - loss: 0.9446 - val_accuracy: 0.6360 - val_loss: 1.0503
Epoch 6/20
704/704 - 323s - 458ms/step - accuracy: 0.6944 - loss: 0.8864 - val_accuracy: 0.7328 - val_loss: 0.7952
Epoch 7/20
704/704 - 319s - 453ms/step - accuracy: 0.7172 - loss: 0.8254 - val_accuracy: 0.7124 - val_loss: 0.8600
Epoch 8/20
704/704 - 275s - 391ms/step - accuracy: 0.7320 - loss: 0.7802 - val_accuracy: 0.7642 - val_loss: 0.7226
Epoch 9/20
704/704 - 277s - 393ms/step - accuracy: 0.7424 - loss: 0.7427 - val_a